# 1. 청구항 작성 데이터셋 만들기 시도 - 0508(금)
### txt 정제 -> 청구항 추출 -> 가상의 상담 note 만들기 -> (가상의 상담노트, 청구항) pair 만들어 보았음

In [ ]:
import os
import re
import json
import openai
from openai import OpenAI
from pathlib import Path
from google.colab import drive

In [ ]:
client = OpenAI(
    api_key="키"
)

In [ ]:
# ==========================================
# 0. 설정 및 API 키
# ==========================================
# 코랩 환경에서 구글 드라이브를 연결(마운트)합니다.
drive.mount('/content/drive')

OPENAI_API_KEY  = "키"

# [수정 1] 알려주신 구글 드라이브 폴더 구조 반영
# '내 드라이브' 최상단에 'final' 폴더가 있다고 가정합니다.
BASE_DRIVE_PATH = "/content/drive/MyDrive/final/extracted_texts"
CATEGORIES = ['G06N', 'G06F', 'G06V', 'G06Q']

# 결과물(JSONL)도 나중에 다운받기 쉽게 드라이브의 final 폴더에 저장합니다.
OUTPUT_JSONL = "/content/drive/MyDrive/final/patent_finetuning_v1.jsonl"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import re

def clean_text(text):
    """
    단락 번호([0018]), 페이지 구분, 등록특허 번호, 페이지 번호 등을 제거하고
    텍스트를 AI 학습용으로 깔끔하게 다듬습니다.
    """
    # 1. 단락 번호 제거 (예: [0018])
    text = re.sub(r'\[\d{4}\]', '', text)

    # 2. 페이지 구분선 제거 (예: --- [페이지 구분] ---)
    # 하이픈(-) 개수나 띄어쓰기가 유동적일 수 있으므로 정규식으로 묶어버립니다.
    text = re.sub(r'-+\s*\[페이지\s*구분\]\s*-+', '', text)

    # 3. 등록특허 번호 제거 (예: 등록특허 10-2951164)
    # '등록특허' 글자와 그 뒤에 오는 '숫자-숫자' 패턴을 날립니다.
    text = re.sub(r'등록특허\s*\d{2}-\d+', '', text)

    # 4. 페이지 번호 제거 (예: - 3 -)
    # 하이픈 기호 사이에 숫자가 있는 패턴을 날립니다.
    text = re.sub(r'-\s*\d+\s*-', '', text)

    # 5. 불필요한 공백 및 빈 줄 정리
    # 위 항목들을 지우고 나면 엔터(\n)가 여러 개 겹쳐서 빈 공간이 붕 뜨게 되는데,
    # 2번 이상 연속된 줄바꿈을 하나의 줄바꿈(\n)으로 압축합니다.
    text = re.sub(r'\n\s*\n', '\n', text)

    return text.strip()

In [ ]:
def extract_patent_elements(text):
    # 1. 발명의 내용 섹션 추출 (괄호 없이 줄바꿈 기준으로 탐색)
    problem_match = re.search(r'해결하려는\s*과제\s*\n(.*?)(?=\n과제의\s*해결\s*수단)', text, re.S)
    solution_match = re.search(r'과제의\s*해결\s*수단\s*\n(.*?)(?=\n발명의\s*효과)', text, re.S)
    effects_match = re.search(r'발명의\s*효과\s*\n(.*?)(?=\n도면의\s*간단한\s*설명|\n발명을\s*실시하기|\n부호의\s*설명|\Z)', text, re.S)

    problem = clean_text(problem_match.group(1)) if problem_match else ""
    solution = clean_text(solution_match.group(1)) if solution_match else ""
    effects = clean_text(effects_match.group(1)) if effects_match else ""

    # 2. 청구범위 추출 (청구범위 시작부터 발명의 설명 시작 전까지)
    claims = []
    claim_section_match = re.search(r'청구범위\s*\n(.*?)(?=\n발명의\s*설명|\n발명의\s*내용|\n발명의\s*상세한\s*설명)', text, re.S)

    if claim_section_match:
        claims_text = claim_section_match.group(1)
        # '청구항 1', '청구항 2' 등을 기준으로 텍스트를 자름
        chunks = re.split(r'\n청구항\s*\d+\s*\n', '\n' + claims_text)

        for chunk in chunks[1:]: # 첫 번째 빈 조각 제외
            cleaned_chunk = clean_text(chunk)
            if cleaned_chunk and cleaned_chunk != '삭제': # '삭제'된 청구항 무시
                claims.append(cleaned_chunk)

    return {
        "problem": problem,
        "solution": solution,
        "effects": effects,
        "claims": claims
    }

In [ ]:
def synthesize_note(elements):
    # 모델이 역할을 더 잘 수행하도록 System 프롬프트 분리
    system_prompt = "당신은 베테랑 변리사입니다. 발명가와의 가상 '상담 Note'를 전문적이고 논리정연하게 작성해 주세요."

    user_prompt = f"""
다음 특허 명세서의 내용을 바탕으로 상담 Note를 작성하세요.
형식은 반드시 아래 4가지 항목을 포함해야 합니다:
[기존 발명 문제점]
[발명 전체 흐름]
[세부 주안점]
[발명의 효과]

명세서 데이터:
- 해결과제: {elements['problem']}
- 해결수단: {elements['solution']}
- 효과: {elements['effects']}
    """

    try:
        # 최신 API 호출 문법 (client.chat.completions.create)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.7
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"API Error: {e}"

In [ ]:
import re

def build_consultation_dataset(claims_list, consultation_note):
    """
    상담 Note를 포함하여 전문 변리사 스타일의 멀티턴 데이터셋을 구축합니다.
    (최종 수정: 엔터/공백 무시 로직 적용 및 카테고리 추출 순서 고정)
    """
    # 1. 카테고리별 설정
    categories = {
        '방법': {
            'end_keywords': ['방법'],
            'replace_text': '방법에 있어서',
            'indep_query': "발명의 시계열적 흐름이나 핵심 로직을 담은 '방법' 독립항을 설계하세요."
        },
        '시스템': {
            'end_keywords': ['시스템', '장치'],
            'replace_text': '시스템에 있어서',
            'indep_query': "장치의 구성 요소와 유기적 결합 관계를 바탕으로 '시스템' 독립항을 설계하세요."
        },
        '매체': {
            'end_keywords': ['매체', '기록매체', '기록 매체'],
            'replace_text': '기록 매체에 있어서',
            'indep_query': "방법 발명을 소프트웨어적으로 구현하여 배포하기 위한 '저장매체' 독립항을 설계하세요."
        }
    }

    # 2. 카테고리 추출 순서 강제 고정 (방법 -> 시스템 -> 매체)
    category_order = ['방법', '시스템', '매체']

    citation_pattern = r'^제?\s*\d+\s*항\s*(?:(?:내지|또는|,|및)\s*제?\s*\d+\s*항)*\s*(?:중\s*어느\s*한\s*항)?에\s*있어서,?'
    system_msg = "당신은 전문 변리사입니다. 제공된 상담 Note를 바탕으로 필수 구성요소만을 포함하여 보호 범위를 극대화한 특허 청구항을 작성해야 합니다."

    final_dataset = []

    # 3. 고정된 순서대로 데이터셋 추출
    for cat_key in category_order:
        info = categories[cat_key]

        # [수정된 필터링 로직]
        # 엔터, 띄어쓰기 등 모든 공백을 제거한 순수 텍스트 상태에서 검사합니다.
        cat_claims = []
        for c in claims_list:
            # \s+ 로 모든 공백/엔터 제거 후 마침표 떼기
            clean_c_for_check = re.sub(r'\s+', '', c).rstrip('.')

            # 끝부분 10글자 이내에 키워드가 있는지 확인
            if any(kw in clean_c_for_check[-10:] for kw in info['end_keywords']):
                cat_claims.append(c) # 원본 데이터는 그대로 보존

        indep_claims = []
        dep_claims = []

        for claim in cat_claims:
            if not re.search(citation_pattern, claim.strip()):
                indep_claims.append(claim.strip())
            else:
                # 종속항 문구 수정
                modified = re.sub(citation_pattern, info['replace_text'], claim.strip()).strip()
                modified = re.sub(r'^,\s*', '', modified)
                dep_claims.append(modified)

        # 독립항이 있어야 데이터셋 구성 가능
        if indep_claims:
            # 1턴: 독립항 설계 (상담 Note 포함)
            user_indep = f"다음 상담 Note를 분석하여 {info['indep_query']}\n\n[상담 Note]\n{consultation_note}"
            assistant_indep = "\n\n".join(indep_claims)

            turn_data = {
                "system": system_msg,
                "turns": [
                    {"user": user_indep, "assistant": assistant_indep}
                ]
            }

            # 2턴: 종속항 설계
            if dep_claims:
                user_dep = f"완벽합니다. 이제 앞서 작성한 '{cat_key}' 독립항의 권리범위를 다층적으로 보호하고 구체화하기 위한 종속항들을 작성하세요."
                assistant_dep = "\n\n".join(dep_claims)
                turn_data["turns"].append({"user": user_dep, "assistant": assistant_dep})

            final_dataset.append(turn_data)

    return final_dataset

In [ ]:
# G06V 폴더의 첫 번째 텍스트 파일을 가져와서 테스트합니다.
test_folder = Path(BASE_DRIVE_PATH) / "G06V"
test_files = list(test_folder.glob("*.txt"))

if test_files:
    test_file_path = test_files[0]
    print(f"테스트 파일: {test_file_path.name}\n" + "="*50)

    with open(test_file_path, 'r', encoding='utf-8') as f:
        sample_text = f.read()

    extracted = extract_patent_elements(sample_text)

    print("\n💡 [해결하려는 과제]:\n")
    print(extracted['problem'])
    print("\n" + "-"*50)

    print("\n💡 [과제의 해결 수단]:\n")
    print(extracted['solution'])
    print("\n" + "-"*50)

    print("\n💡 [발명의 효과]:\n")
    print(extracted['effects'])
    print("\n" + "="*50)

    print(f"\n📌 [추출된 청구항 총 {len(extracted['claims'])}개]")
    for i, claim in enumerate(extracted['claims']):
        print(f"\n▶ [청구항 {i+1}]:\n{claim}")
else:
    print("경로에 텍스트 파일이 없습니다. 폴더 경로를 확인해 주세요.")

테스트 파일: 1020250172807.txt

💡 [해결하려는 과제]:

따라서, 본 발명의 제1 목적은 차량의 현장의 영상데이터를 게이트웨이를 통해 클라우드로 전송하고, 클라우드
서버에서 스트립 기반의 이벤트 트리거 전처리 기술과, 번호판의 형태적 특징에 최적화된 딥러닝 아키텍처를 유
기적으로 결합하여, 다중 채널 환경에서의 연산 효율과 인식 정확도를 동시에 대폭 향상시킨 번호판 인식 시스
템을 제공하는데 있다.
 또한, 본 발명의 제2 목적은 영상데이터의 프레임에서 스트립 영역을 설정하고, 스트립 영역의 신호 변화량만
을 분석하는 경량 연산을 수행하여 딥러닝 추론이 필요한 이벤트 프레임을 선별하며, 선별된 이벤트 프레임만
번호판의 문자를 인식하여 전체 연산 부하를 저감하고 다중 채널 처리량을 증대시키는 번호판 인식 방법을 제공
하는데 있다.

--------------------------------------------------

💡 [과제의 해결 수단]:

상술한 본 발명의 제1 목적을 달성하기 위하여, 본 발명의 일 실시예에서는 주행 중인 차량의 영상을 촬영해 영
상데이터를 생성하는 감시 카메라와, 상기 영상데이터를 통신 네트워크를 통해 전송하는 게이트웨이, 및 상기
게이트웨이로부터 수신한 영상데이터의 프레임에서 스트립 영역을 설정하고, 상기 스트립 영역의 신호 변화량을
분석해 차량이 통과하는 이벤트 프레임을 감지해 추출하는 트리아지 코어, 및 번호판 문자열의 수평적 특징과
수직적 특징을 동시에 강조하도록 설계된 딥러닝 모델을 이용하여 상기 이벤트 프레임 내의 번호판 문자를 검지
하고 인식하는 코그니션 코어가 포함된 클라우드 서버를 포함하는 클라우드 기반 전후방 번호판 인식 시스템을
제공한다.
또한, 본 발명의 제2 목적을 달성하기 위하여, 본 발명의 일 실시예에서는 클라우드 서버에서 다중 채널의 영상
데이터를 처리하여 번호판을 인식하는 방법에 있어서, 감시 카메라 및 게이트웨이로부터 제공된 각 채널의 영상
데이터에 대해, 클라우드 서버의 트리아지

In [ ]:
if test_files:
    test_file_path = test_files[0]
    print(f"📄 테스트 파일: {test_file_path.name}")
    print("⏳ 1단계: 텍스트에서 명세서 데이터를 추출합니다...")

    # 파일 읽기
    with open(test_file_path, 'r', encoding='utf-8') as f:
        sample_text = f.read()

    # 데이터 추출
    extracted_data = extract_patent_elements(sample_text)

    # 추출이 잘 되었는지 체크 (빈 데이터면 OpenAI 호출 안 함)
    if not extracted_data['problem']:
        print("❌ 추출 실패: '해결하려는 과제'를 찾지 못했습니다. 텍스트 구조를 확인해주세요.")
    else:
        print("✅ 추출 완료! OpenAI API를 호출하여 상담 Note를 생성합니다...")
        print("=" * 50)

        # OpenAI 합성 실행
        final_note = synthesize_note(extracted_data)

        # 최종 결과 출력
        print(final_note)
        print("=" * 50)
else:
    print("❌ 경로에 텍스트 파일이 없습니다. 구글 드라이브 마운트 및 폴더 경로를 확인해 주세요.")

📄 테스트 파일: 1020250172807.txt
⏳ 1단계: 텍스트에서 명세서 데이터를 추출합니다...
✅ 추출 완료! OpenAI API를 호출하여 상담 Note를 생성합니다...
### 상담 Note

#### [기존 발명 문제점]
기존의 차량 번호판 인식 시스템은 현장에서 고부하의 연산을 수행하기 위해 비싼 현장 제어기를 설치해야 했습니다. 이로 인해 초기 도입 비용 및 운영 비용이 증가하였고, 다양한 환경적 요인(역광, 비, 오염 등)에서도 번호판 인식의 정확성이 떨어지는 문제가 있었습니다. 또한, 다중 채널의 영상 데이터를 처리하는 과정에서 연산 부하가 증가하여 전체 처리량이 저하되는 단점이 존재했습니다.

#### [발명 전체 흐름]
본 발명은 차량의 현장 영상 데이터를 게이트웨이를 통해 클라우드로 전송하고, 클라우드 서버에서 스트립 기반의 이벤트 트리거 전처리 기술과 최적화된 딥러닝 아키텍처를 결합하여 번호판 인식을 수행하는 시스템을 제안합니다. 구체적으로, 감시 카메라가 영상 데이터를 생성하고, 게이트웨이가 이를 클라우드 서버로 전송합니다. 클라우드 서버의 트리아지 코어는 영상 데이터에서 이벤트 프레임을 선별하고, 코그니션 코어가 이 프레임에 대해 번호판 문자를 인식합니다. 이를 통해 불필요한 연산을 줄이고, 효율적으로 번호판 인식을 수행합니다.

#### [세부 주안점]
1. **스트립 영역 설정 및 경량 연산:** 클라우드 서버의 트리아지 코어는 영상 데이터의 프레임에서 스트립 영역을 설정하고, 해당 영역의 신호 변화량을 분석하여 이벤트 프레임을 선별합니다. 이 경량 연산을 통해 딥러닝 추론이 필요한 프레임을 효율적으로 찾아냅니다.
   
2. **딥러닝 모델 최적화:** 코그니션 코어는 번호판의 수평적 및 수직적 특징을 동시에 강조하도록 설계된 딥러닝 모델을 사용하여, 다양한 환경 조건에서도 높은 정확도로 번호판을 검출합니다.

3. **클라우드 기반 처리:** 모든 고부하 연산을 클라우드 서버에서 중앙 집중 처리하여, 현장에서는 감시 카

In [ ]:
print("\n⏳ 2단계: 멀티턴 데이터셋을 구축 중입니다...")
dataset = build_consultation_dataset(extracted_data['claims'], final_note)

print(f"✅ 구축 완료! 생성된 데이터셋 수: {len(dataset)}개 (카테고리별 그룹)")
print("=" * 50)

# 첫 번째 데이터셋 샘플 출력
if dataset:
    sample = dataset[0]
    print(f"📌 [SYSTEM]: {sample['system']}")
    for i, turn in enumerate(sample['turns']):
        print(f"\n[TURN {i+1} USER]: {turn['user'][:100]}...") # 너무 기니까 앞부분만
        print(f"\n[TURN {i+1} AGENT]: {turn['assistant'][:100]}...")
print("=" * 50)


⏳ 2단계: 멀티턴 데이터셋을 구축 중입니다...
✅ 구축 완료! 생성된 데이터셋 수: 2개 (카테고리별 그룹)
📌 [SYSTEM]: 당신은 전문 변리사입니다. 제공된 상담 Note를 바탕으로 필수 구성요소만을 포함하여 보호 범위를 극대화한 특허 청구항을 작성해야 합니다.

[TURN 1 USER]: 다음 상담 Note를 분석하여 발명의 시계열적 흐름이나 핵심 로직을 담은 '방법' 독립항을 설계하세요.

[상담 Note]
### 상담 Note

#### [기존 발명 문제점]
기...

[TURN 1 AGENT]: 클라우드 서버에서 다중 채널의 영상데이터를 처리하여 번호판을 인식하는 방법에 있어서, 
감시 카메라로부터 제공된 각 채널의 영상데이터에 대해, 클라우드 서버의 트리아지 코어가 상기...

[TURN 2 USER]: 완벽합니다. 이제 앞서 작성한 '방법' 독립항의 권리범위를 다층적으로 보호하고 구체화하기 위한 종속항들을 작성하세요....

[TURN 2 AGENT]: 방법에 있어서 상기 프레임 선별단계는
상기 프레임의 해상도를 축소하는 해상도 축소과정과,
해상도가 축소된 프레임에서 차량이 통행하는 영역에 감지선을 도입하는 감지선 도입과정과,
상...


In [ ]:
print(f"DEBUG: 추출된 첫 번째 청구항 예시: {extracted_data['claims'][0] if extracted_data['claims'] else '없음'}")

DEBUG: 추출된 첫 번째 청구항 예시: 주행 중인 차량의 영상을 촬영해 영상데이터를 생성하는 감시 카메라; 및
상기 감시 카메라로부터 제공된 영상데이터의 프레임에서 스트립 영역을 설정하고, 상기 스트립 영역의 신호 변
화량을 분석해 차량이 통과하는 이벤트 프레임을 감지해 추출하는 트리아지 코어, 및 번호판 문자열의 수평적
특징과 수직적 특징을 동시에 강조하도록 설계된 딥러닝 모델을 이용하여 상기 이벤트 프레임 내의 번호판 문자
를 검지하고 인식하는 코그니션 코어가 포함된 클라우드 서버를 포함하며,
상기 트리아지 코어는 상기 스트립 영역의 신호 변화량에 기반한 이벤트 감지 동작과 독립적으로, 이벤트 감지
누락에 대한 안정성을 확보하도록 사전에 설정된 주기에 따라 주기적으로 이벤트 프레임을 강제 추출하여 상기
코그니션 코어로 제공하는 주기적 샘플링 기능을 포함하는 클라우드 기반 전후방 번호판 인식 시스템.


In [ ]:
import json
import time
from pathlib import Path

# 1. Google Drive 마운트 (코랩 환경인 경우 실행)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Colab 환경이 아닙니다. 로컬 경로를 사용합니다.")

# 2. 경로 설정
# 드라이브 최상단(MyDrive) 하위에 final 폴더가 있다고 가정합니다.
# 실제 경로가 다를 경우 '/content/drive/MyDrive/...' 부분을 알맞게 수정해 주세요.
BASE_DIR = Path('/content/drive/MyDrive/final/extracted_texts')
TARGET_FOLDERS = ['G06F', 'G06N', 'G06Q', 'G06V']

# 3. 결과물이 저장될 JSONL 파일 경로
output_jsonl_path = Path('/content/drive/MyDrive/final/patent_multiturn_dataset.jsonl')

print("🚀 데이터셋 추출 및 JSONL 저장 프로세스를 시작합니다...")

# 기존 파일 덮어쓰기 방지를 위해 파일을 새로 생성(초기화)합니다.
# 만약 중간에 끊겨서 이어서 작업해야 한다면 이 부분(블록)을 주석 처리하세요.
with open(output_jsonl_path, 'w', encoding='utf-8') as f:
    pass

total_saved_items = 0

for folder_name in TARGET_FOLDERS:
    folder_path = BASE_DIR / folder_name

    if not folder_path.exists():
        print(f"⚠️ 폴더를 찾을 수 없습니다: {folder_path}")
        continue

    txt_files = list(folder_path.glob("*.txt"))
    print(f"\n📂 [{folder_name}] 폴더 처리 시작 (총 {len(txt_files)}개 파일 발견)")

    for file_path in txt_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()

            # 1. 데이터 추출
            extracted = extract_patent_elements(text)
            if not extracted['claims']:
                continue # 청구항이 추출되지 않은 파일은 스킵

            # 2. 상담 Note 생성 (OpenAI API 호출)
            # 💡 중요: OpenAI API Rate Limit 방지를 위해 요청 사이에 약간의 휴식을 줍니다.
            time.sleep(0.5)
            note = synthesize_note(extracted)

            # API 에러 발생 시 건너뛰기
            if "API Error" in note:
                print(f"❌ API 에러 발생 ({file_path.name}) - 스킵합니다.")
                continue

            # 3. 멀티턴 데이터셋 변환
            dataset = build_consultation_dataset(extracted['claims'], note)

            # 4. JSONL 파일에 실시간으로 기록 (Append 모드 'a')
            if dataset:
                with open(output_jsonl_path, 'a', encoding='utf-8') as outfile:
                    for data_item in dataset:
                        # ensure_ascii=False: 한글이 유니코드(예: \uc548)로 깨지는 것을 방지
                        json_line = json.dumps(data_item, ensure_ascii=False)
                        outfile.write(json_line + '\n')
                        total_saved_items += 1

        except Exception as e:
            print(f"⚠️ 파일 처리 중 예기치 않은 오류 발생 ({file_path.name}): {e}")

print("\n" + "="*50)
print(f"✅ 모든 작업 완료! 총 {total_saved_items}개의 학습 데이터가 성공적으로 구축되었습니다.")
print(f"📁 최종 파일 저장 위치: {output_jsonl_path}")
print("="*50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 데이터셋 추출 및 JSONL 저장 프로세스를 시작합니다...

📂 [G06F] 폴더 처리 시작 (총 110개 파일 발견)

📂 [G06N] 폴더 처리 시작 (총 143개 파일 발견)

📂 [G06Q] 폴더 처리 시작 (총 192개 파일 발견)

📂 [G06V] 폴더 처리 시작 (총 97개 파일 발견)

✅ 모든 작업 완료! 총 895개의 학습 데이터가 성공적으로 구축되었습니다.
📁 최종 파일 저장 위치: /content/drive/MyDrive/final/patent_multiturn_dataset.jsonl


# 2. 검토 agent 데이터 셋 만들기 - 0511(월)

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time
from google.colab import files

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time
from google.colab import files

# 142개 출원번호 (변리사님 리스트 그대로)
app_numbers = [
    "1020257019800", "1020250155326", "1020250115002", "1020250077859", "1020250077843",
    "1020250068220", "1020250051746", "1020250045991", "1020250034744", "1020250027252",
    "1020247023700", "1020247023307", "1020247011628", "1020240196542", "1020240173044",
    "1020240158222", "1020240152277", "1020240146684", "1020240138555", "1020240081489",
    "1020240058416", "1020240026694", "1020237015600", "1020237000315", "1020230150449",
    "1020230150441", "1020230142119", "1020230134631", "1020230131860", "1020230118769",
    "1020230118745", "1020230102592", "1020230078104", "1020230061000", "1020230040630",
    "1020230036605", "1020230033546", "1020230025066", "1020230024571", "1020227027088",
    "1020227026698", "1020227017608", "1020227015892", "1020227015590", "1020227010826",
    "1020227009936", "1020227009700", "1020227009490", "1020227009135", "1020227008703",
    "1020227008374", "1020227000401", "1020220148408", "1020220147688", "1020220140197",
    "1020220131207", "1020220129423", "1020220126844", "1020220104321", "1020220104033",
    "1020220091980", "1020220090887", "1020220088711", "1020220084314", "1020220072515",
    "1020220071866", "1020220057805", "1020220033925", "1020220032074", "1020220026671",
    "1020220002233", "1020220002232", "1020220002231", "1020217033033", "1020217028715",
    "1020217026855", "1020217023623", "1020217018412", "1020217017878", "1020217011467",
    "1020210172385", "1020210171633", "1020210168628", "1020210168032", "1020210161676",
    "1020210154767", "1020210134063", "1020210127454", "1020210117172", "1020210112340",
    "1020210085762", "1020210085431", "1020210068892", "1020210057704", "1020210046441",
    "1020210037975", "1020207036632", "1020207035847", "1020207035844", "1020207035843",
    "1020207032320", "1020207024742", "1020207018024", "1020207014462", "1020207004284",
    "1020200168183", "1020200163675", "1020200161286", "1020200148939", "1020200126707",
    "1020200120202", "1020200103968", "1020200093243", "1020200092399", "1020200059705",
    "1020200004518", "1020200001729", "1020200001714", "1020197037891", "1020197026115",
    "1020197015991", "1020197013953", "1020197012085", "1020190164745", "1020190101984",
    "1020190098351", "1020190093807", "1020190090963", "1020187037824", "1020180125990",
    "1020180123300", "1020180116750", "1020180063015", "1020180031088", "1020180025971",
    "1020177006950", "1020170155897", "1020170144234", "1020170122363", "1020170097848",
    "1020170083779", "1020170054474", "1020170003348"
]

API_KEY = "키" # API 키만 수정해주세요!
BASE_URL = "http://plus.kipris.or.kr/openapi/rest/IntermediateDocumentOPService/advancedSearchInfo"

results = []
print(f"조회 시작: 총 {len(app_numbers)}건\n")

for idx, app_num in enumerate(app_numbers):
    params = {
        "applicationNumber": app_num,
        "patent": "true",
        "accessKey": API_KEY
    }

    try:
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status()

        root = ET.fromstring(response.content)
        found_oa = False

        # API가 반환하는 <advancedSearchInfo> 태그 자체가 곧 통지서 데이터입니다.
        for item in root.findall('.//advancedSearchInfo'):
            found_oa = True

            # 필터링 없이 곧바로 데이터를 수집합니다.
            results.append({
                "출원번호": item.findtext('applicationNumber'),
                "발송번호": item.findtext('sendNumber'),
                "발송일자": item.findtext('sendDate'),
                "발명의명칭": item.findtext('title'), # 문서명이 아닌 특허의 명칭
                "PDF링크": item.findtext('filePath') # 핵심 타겟: PDF 다운로드 주소
            })

        if found_oa:
            print(f"[{idx+1}/{len(app_numbers)}] ✅ {app_num}: 의견제출통지서(OA) 확보!")
        else:
            print(f"[{idx+1}/{len(app_numbers)}] ❌ {app_num}: OA 없음 (스트레이트 등록이거나 공개상태)")

    except Exception as e:
        print(f"[{idx+1}/{len(app_numbers)}] ⚠️ {app_num}: 에러 발생 ({e})")

    time.sleep(0.5)

# 결과 정리 및 다운로드
df = pd.DataFrame(results)

if not df.empty:
    filename = "KIPO_OA_PDF_LINKS_FINAL.xlsx"
    df.to_excel(filename, index=False)
    print(f"\n축하합니다! 총 {len(df)}건의 PDF 링크를 엑셀로 저장했습니다.")
    files.download(filename)
else:
    print("\n추출된 데이터가 없습니다. (API 키 오류이거나, 정말로 해당 번호들에 통지서가 없을 확률이 높습니다.)")

조회 시작: 총 143건

[1/143] ✅ 1020257019800: 의견제출통지서(OA) 확보!
[2/143] ❌ 1020250155326: OA 없음 (스트레이트 등록이거나 공개상태)
[3/143] ✅ 1020250115002: 의견제출통지서(OA) 확보!
[4/143] ✅ 1020250077859: 의견제출통지서(OA) 확보!
[5/143] ✅ 1020250077843: 의견제출통지서(OA) 확보!
[6/143] ✅ 1020250068220: 의견제출통지서(OA) 확보!
[7/143] ❌ 1020250051746: OA 없음 (스트레이트 등록이거나 공개상태)
[8/143] ✅ 1020250045991: 의견제출통지서(OA) 확보!
[9/143] ✅ 1020250034744: 의견제출통지서(OA) 확보!
[10/143] ✅ 1020250027252: 의견제출통지서(OA) 확보!
[11/143] ✅ 1020247023700: 의견제출통지서(OA) 확보!
[12/143] ✅ 1020247023307: 의견제출통지서(OA) 확보!
[13/143] ✅ 1020247011628: 의견제출통지서(OA) 확보!
[14/143] ✅ 1020240196542: 의견제출통지서(OA) 확보!
[15/143] ✅ 1020240173044: 의견제출통지서(OA) 확보!
[16/143] ❌ 1020240158222: OA 없음 (스트레이트 등록이거나 공개상태)
[17/143] ✅ 1020240152277: 의견제출통지서(OA) 확보!
[18/143] ✅ 1020240146684: 의견제출통지서(OA) 확보!
[19/143] ✅ 1020240138555: 의견제출통지서(OA) 확보!
[20/143] ✅ 1020240081489: 의견제출통지서(OA) 확보!
[21/143] ✅ 1020240058416: 의견제출통지서(OA) 확보!
[22/143] ✅ 1020240026694: 의견제출통지서(OA) 확보!
[23/143] ✅ 1020237015600: 의견제출통지서(OA) 확보!
[

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import requests
import io
import time
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload

In [ ]:
auth.authenticate_user()
drive_service = build('drive', 'v3')

In [ ]:
# 2. 설정 정보 (사용자 입력 기반)
PARENT_FOLDER_ID = "final폴더" # 공유해주신 상위 폴더 ID
EXCEL_FILE = "KIPO_OA_PDF_LINKS_FINAL.xlsx"          # 아까 생성된 엑셀 파일명
SUBFOLDER_NAME = "심사관"

In [ ]:
def create_drive_folder(folder_name, parent_id):
    file_metadata = {
        'name': folder_name,
        'mimeType': 'application/vnd.google-apps.folder',
        'parents': [parent_id]
    }
    folder = drive_service.files().create(body=file_metadata, fields='id').execute()
    return folder.get('id')

In [ ]:
try:
    # '심사관' 폴더 생성 및 ID 확보
    target_folder_id = create_drive_folder(SUBFOLDER_NAME, PARENT_FOLDER_ID)
    print(f"✅ 구글 드라이브에 '{SUBFOLDER_NAME}' 폴더 생성 완료 (ID: {target_folder_id})")

    # 엑셀 데이터 로드
    df = pd.read_excel(EXCEL_FILE)
    print(f"총 {len(df)}개의 파일을 처리합니다...\n")

    for index, row in df.iterrows():
        app_num = str(row['출원번호'])
        pdf_url = row['PDF링크']

        # 링크가 없는 경우 제외
        if pd.isna(pdf_url) or pdf_url == "링크없음":
            print(f"[{index+1}] {app_num}: 다운로드 가능한 링크가 없습니다.")
            continue

        try:
            # KIPRIS 서버에서 PDF 다운로드
            # Referer나 User-Agent가 필요한 경우를 대비해 기본 헤더 추가
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(pdf_url, headers=headers, timeout=30)

            if response.status_code == 200:
                # 구글 드라이브에 업로드할 메타데이터 설정
                file_metadata = {
                    'name': f"{app_num}.pdf", # 파일명을 출원번호.pdf로 설정
                    'parents': [target_folder_id]
                }

                # 메모리 내 바이너리 데이터를 드라이브로 전송
                media = MediaIoBaseUpload(io.BytesIO(response.content), mimetype='application/pdf')
                drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()
                print(f"[{index+1}] ✅ {app_num}.pdf 업로드 성공")
            else:
                print(f"[{index+1}] ❌ {app_num}: 다운로드 실패 (상태 코드: {response.status_code})")

        except Exception as e:
            print(f"[{index+1}] ⚠️ {app_num}: 처리 중 에러 발생 - {e}")

        # 서버 부하 방지를 위해 짧은 휴식
        time.sleep(0.3)

    print("\n✨ 모든 작업이 완료되었습니다. 구글 드라이브 폴더를 확인해 보세요!")

except Exception as e:
    print(f"🛑 폴더 생성 또는 파일 처리 중 치명적 에러 발생: {e}")

✅ 구글 드라이브에 '심사관' 폴더 생성 완료 (ID: 1JRFzcOAw_yoDBxYax69HkzoN09iv_D7h)
총 138개의 파일을 처리합니다...

[1] ✅ 1020257019800.pdf 업로드 성공
[2] ✅ 1020250115002.pdf 업로드 성공
[3] ✅ 1020250077859.pdf 업로드 성공
[4] ✅ 1020250077843.pdf 업로드 성공
[5] ✅ 1020250068220.pdf 업로드 성공
[6] ✅ 1020250045991.pdf 업로드 성공
[7] ✅ 1020250034744.pdf 업로드 성공
[8] ✅ 1020250027252.pdf 업로드 성공
[9] ✅ 1020247023700.pdf 업로드 성공
[10] ✅ 1020247023307.pdf 업로드 성공
[11] ✅ 1020247011628.pdf 업로드 성공
[12] ✅ 1020240196542.pdf 업로드 성공
[13] ✅ 1020240173044.pdf 업로드 성공
[14] ✅ 1020240152277.pdf 업로드 성공
[15] ✅ 1020240146684.pdf 업로드 성공
[16] ✅ 1020240138555.pdf 업로드 성공
[17] ✅ 1020240081489.pdf 업로드 성공
[18] ✅ 1020240058416.pdf 업로드 성공
[19] ✅ 1020240026694.pdf 업로드 성공
[20] ✅ 1020237015600.pdf 업로드 성공
[21] ✅ 1020237000315.pdf 업로드 성공
[22] ✅ 1020230142119.pdf 업로드 성공
[23] ✅ 1020230142119.pdf 업로드 성공
[24] ✅ 1020230134631.pdf 업로드 성공
[25] ✅ 1020230131860.pdf 업로드 성공
[26] ✅ 1020230118769.pdf 업로드 성공
[27] ✅ 1020230118745.pdf 업로드 성공
[28] ✅ 1020230102592.pdf 업로드 성공
[29] ✅ 1020230078104.pdf 

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time
from google.colab import files

# 142개 출원번호 전체 리스트
app_numbers = [
    "1020257019800", "1020250155326", "1020250115002", "1020250077859", "1020250077843",
    "1020250068220", "1020250051746", "1020250045991", "1020250034744", "1020250027252",
    "1020247023700", "1020247023307", "1020247011628", "1020240196542", "1020240173044",
    "1020240158222", "1020240152277", "1020240146684", "1020240138555", "1020240081489",
    "1020240058416", "1020240026694", "1020237015600", "1020237000315", "1020230150449",
    "1020230150441", "1020230142119", "1020230134631", "1020230131860", "1020230118769",
    "1020230118745", "1020230102592", "1020230078104", "1020230061000", "1020230040630",
    "1020230036605", "1020230033546", "1020230025066", "1020230024571", "1020227027088",
    "1020227026698", "1020227017608", "1020227015892", "1020227015590", "1020227010826",
    "1020227009936", "1020227009700", "1020227009490", "1020227009135", "1020227008703",
    "1020227008374", "1020227000401", "1020220148408", "1020220147688", "1020220140197",
    "1020220131207", "1020220129423", "1020220126844", "1020220104321", "1020220104033",
    "1020220091980", "1020220090887", "1020220088711", "1020220084314", "1020220072515",
    "1020220071866", "1020220057805", "1020220033925", "1020220032074", "1020220026671",
    "1020220002233", "1020220002232", "1020220002231", "1020217033033", "1020217028715",
    "1020217026855", "1020217023623", "1020217018412", "1020217017878", "1020217011467",
    "1020210172385", "1020210171633", "1020210168628", "1020210168032", "1020210161676",
    "1020210154767", "1020210134063", "1020210127454", "1020210117172", "1020210112340",
    "1020210085762", "1020210085431", "1020210068892", "1020210057704", "1020210046441",
    "1020210037975", "1020207036632", "1020207035847", "1020207035844", "1020207035843",
    "1020207032320", "1020207024742", "1020207018024", "1020207014462", "1020207004284",
    "1020200168183", "1020200163675", "1020200161286", "1020200148939", "1020200126707",
    "1020200120202", "1020200103968", "1020200093243", "1020200092399", "1020200059705",
    "1020200004518", "1020200001729", "1020200001714", "1020197037891", "1020197026115",
    "1020197015991", "1020197013953", "1020197012085", "1020190164745", "1020190101984",
    "1020190098351", "1020190093807", "1020190090963", "1020187037824", "1020180125990",
    "1020180123300", "1020180116750", "1020180063015", "1020180031088", "1020180025971",
    "1020177006950", "1020170155897", "1020170144234", "1020170122363", "1020170097848",
    "1020170083779", "1020170054474", "1020170003348"
]

API_KEY = "키" # KIPRIS API 키 입력
BASE_URL = "http://plus.kipris.or.kr/openapi/rest/ClaimsChangeHistoryService/amendmentHistoryInfo"

results = []
print(f"조회 시작: 총 {len(app_numbers)}건\n")

for idx, app_num in enumerate(app_numbers):
    params = {
        "applicationNumber": app_num,
        "accessKey": API_KEY
    }

    try:
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status()

        root = ET.fromstring(response.content)
        found_history = False

        # XML 구조에 맞춰 <amendmentHistoryInfo> 태그를 순회
        for item in root.findall('.//amendmentHistoryInfo'):
            found_history = True

            results.append({
                "출원번호": item.findtext('applicationNumber'),
                "일련번호": item.findtext('receiptSendSerialNumber'),
                "접수발송번호": item.findtext('receiptSendNumber'),
                "접수발송일자": item.findtext('receiptSendDate'),
                "문서코드": item.findtext('receiptSendDocumentCode'),
                "문서명": item.findtext('receiptSendDocumentName')
            })

        if found_history:
            print(f"[{idx+1}/{len(app_numbers)}] ✅ {app_num}: 변동이력 확보 완료")
        else:
            print(f"[{idx+1}/{len(app_numbers)}] ❌ {app_num}: 이력 없음 (출원 상태 등)")

    except Exception as e:
        print(f"[{idx+1}/{len(app_numbers)}] ⚠️ {app_num}: API 에러 발생 ({e})")

    time.sleep(0.5) # API 호출 제한 방지용 딜레이

# 데이터프레임 변환 및 엑셀 다운로드 처리
df = pd.DataFrame(results)

if not df.empty:
    filename = "KIPO_CLAIM_HISTORY_LIST.xlsx"
    df.to_excel(filename, index=False)
    print(f"\n작업 완료! 총 {len(df)}건의 이력 데이터를 엑셀로 저장했습니다.")
    files.download(filename)
else:
    print("\n추출된 이력 데이터가 없습니다. API 키를 다시 한 번 확인해 주세요.")

조회 시작: 총 143건

[1/143] ✅ 1020257019800: 변동이력 확보 완료
[2/143] ❌ 1020250155326: 이력 없음 (출원 상태 등)
[3/143] ✅ 1020250115002: 변동이력 확보 완료
[4/143] ❌ 1020250077859: 이력 없음 (출원 상태 등)
[5/143] ❌ 1020250077843: 이력 없음 (출원 상태 등)
[6/143] ✅ 1020250068220: 변동이력 확보 완료
[7/143] ❌ 1020250051746: 이력 없음 (출원 상태 등)
[8/143] ✅ 1020250045991: 변동이력 확보 완료
[9/143] ✅ 1020250034744: 변동이력 확보 완료
[10/143] ✅ 1020250027252: 변동이력 확보 완료
[11/143] ✅ 1020247023700: 변동이력 확보 완료
[12/143] ✅ 1020247023307: 변동이력 확보 완료
[13/143] ✅ 1020247011628: 변동이력 확보 완료
[14/143] ✅ 1020240196542: 변동이력 확보 완료
[15/143] ✅ 1020240173044: 변동이력 확보 완료
[16/143] ✅ 1020240158222: 변동이력 확보 완료
[17/143] ❌ 1020240152277: 이력 없음 (출원 상태 등)
[18/143] ✅ 1020240146684: 변동이력 확보 완료
[19/143] ✅ 1020240138555: 변동이력 확보 완료
[20/143] ✅ 1020240081489: 변동이력 확보 완료
[21/143] ✅ 1020240058416: 변동이력 확보 완료
[22/143] ✅ 1020240026694: 변동이력 확보 완료
[23/143] ✅ 1020237015600: 변동이력 확보 완료
[24/143] ✅ 1020237000315: 변동이력 확보 완료
[25/143] ✅ 1020230150449: 변동이력 확보 완료
[26/143] ✅ 1020230150441: 변동이력 확보 완료
[27/143

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests

# 샘플 번호 하나로 테스트
test_num = "1020257019800"
API_KEY = "키"
BASE_URL = "http://plus.kipris.or.kr/openapi/rest/FullTextService/getPublicationFullTextInfo"

response = requests.get(BASE_URL, params={"applicationNumber": test_num, "accessKey": API_KEY})

print(f"상태 코드: {response.status_code}")
print("--- 서버 응답 앞부분 (이걸 복사해서 저에게 보여주세요) ---")
print(response.text[:500])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time
from google.colab import files

# 1. 143개 출원번호 리스트
app_numbers = [
    "1020257019800", "1020250155326", "1020250115002", "1020250077859", "1020250077843",
    "1020250068220", "1020250051746", "1020250045991", "1020250034744", "1020250027252",
    "1020247023700", "1020247023307", "1020247011628", "1020240196542", "1020240173044",
    "1020240158222", "1020240152277", "1020240146684", "1020240138555", "1020240081489",
    "1020240058416", "1020240026694", "1020237015600", "1020237000315", "1020230150449",
    "1020230150441", "1020230142119", "1020230134631", "1020230131860", "1020230118769",
    "1020230118745", "1020230102592", "1020230078104", "1020230061000", "1020230040630",
    "1020230036605", "1020230033546", "1020230025066", "1020230024571", "1020227027088",
    "1020227026698", "1020227017608", "1020227015892", "1020227015590", "1020227010826",
    "1020227009936", "1020227009700", "1020227009490", "1020227009135", "1020227008703",
    "1020227008374", "1020227000401", "1020220148408", "1020220147688", "1020220140197",
    "1020220131207", "1020220129423", "1020220126844", "1020220104321", "1020220104033",
    "1020220091980", "1020220090887", "1020220088711", "1020220084314", "1020220072515",
    "1020220071866", "1020220057805", "1020220033925", "1020220032074", "1020220026671",
    "1020220002233", "1020220002232", "1020220002231", "1020217033033", "1020217028715",
    "1020217026855", "1020217023623", "1020217018412", "1020217017878", "1020217011467",
    "1020210172385", "1020210171633", "1020210168628", "1020210168032", "1020210161676",
    "1020210154767", "1020210134063", "1020210127454", "1020210117172", "1020210112340",
    "1020210085762", "1020210085431", "1020210068892", "1020210057704", "1020210046441",
    "1020210037975", "1020207036632", "1020207035847", "1020207035844", "1020207035843",
    "1020207032320", "1020207024742", "1020207018024", "1020207014462", "1020207004284",
    "1020200168183", "1020200163675", "1020200161286", "1020200148939", "1020200126707",
    "1020200120202", "1020200103968", "1020200093243", "1020200092399", "1020200059705",
    "1020200004518", "1020200001729", "1020200001714", "1020197037891", "1020197026115",
    "1020197015991", "1020197013953", "1020197012085", "1020190164745", "1020190101984",
    "1020190098351", "1020190093807", "1020190090963", "1020187037824", "1020180125990",
    "1020180123300", "1020180116750", "1020180063015", "1020180031088", "1020180025971",
    "1020177006950", "1020170155897", "1020170144234", "1020170122363", "1020170097848",
    "1020170083779", "1020170054474", "1020170003348"
]

# 2. 제공해주신 API 주소 (kipo-api 방식)
SERVICE_KEY = "키"  # 주의: 이 방식은 accessKey가 아니라 ServiceKey 파라미터를 씁니다.
BASE_URL = "http://plus.kipris.or.kr/kipo-api/kipi/patUtiModInfoSearchSevice/getPubFullTextInfoSearch"

results = []
print(f"공개공보 전용 조회를 시작합니다 (총 {len(app_numbers)}건)\n")

for idx, app_num in enumerate(app_numbers):
    params = {
        "applicationNumber": app_num,
        "ServiceKey": SERVICE_KEY
    }

    try:
        response = requests.get(BASE_URL, params=params, timeout=30)

        # HTML 에러 방어
        if not response.text.strip().startswith("<?xml") and not response.text.strip().startswith("<response"):
            print(f"[{idx+1}/{len(app_numbers)}] (코드: N/A) ❌ {app_num}: 서버 비정상 응답 (XML 아님)")
            continue

        root = ET.fromstring(response.content)

        # 요구사항 반영: header 안의 resultCode 파싱
        result_code_node = root.find('.//header/resultCode')
        if result_code_node is None:
            result_code_node = root.find('.//resultCode') # 혹시 몰라 전체에서 한번 더 찾기

        code_text = result_code_node.text if result_code_node is not None else "N/A"

        # 코드별 처리
        if code_text == "00":
            # 정상 서비스일 경우 body > item > path 파싱
            path_node = root.find('.//body/item/path')
            if path_node is None:
                path_node = root.find('.//path') # 방어 로직

            if path_node is not None and path_node.text:
                results.append({"출원번호": app_num, "공개공보링크": path_node.text})
                print(f"[{idx+1}/{len(app_numbers)}] (코드: {code_text}) ✅ {app_num}: 링크 확보!")
            else:
                print(f"[{idx+1}/{len(app_numbers)}] (코드: {code_text}) ❌ {app_num}: 데이터 없음 (태그는 정상이나 값이 비어있음)")

        elif code_text == "22":
            print(f"[{idx+1}/{len(app_numbers)}] (코드: {code_text}) 🛑 오늘 쿼터를 모두 소진했습니다! 조회를 중단합니다.")
            break
        elif code_text == "30":
            print(f"[{idx+1}/{len(app_numbers)}] (코드: {code_text}) 🛑 키 권한 오류! (등록되지 않은 키이거나 서비스 미승인)")
            break
        elif code_text == "31":
            print(f"[{idx+1}/{len(app_numbers)}] (코드: {code_text}) ⚠️ {app_num}: 타임아웃 지연 (DEADLINE_HAS_EXPIRED_ERROR)")
        else:
            # 기타 에러 코드
            msg_node = root.find('.//resultMsg')
            msg_text = msg_node.text if msg_node is not None else "알 수 없는 에러"
            print(f"[{idx+1}/{len(app_numbers)}] (코드: {code_text}) ❓ {app_num}: 기타 에러 - {msg_text}")

    except requests.exceptions.Timeout:
        print(f"[{idx+1}/{len(app_numbers)}] (코드: N/A) ⚠️ {app_num}: HTTP 요청 타임아웃 발생")
    except ET.ParseError as e:
        print(f"[{idx+1}/{len(app_numbers)}] (코드: N/A) ⚠️ {app_num}: XML 파싱 에러 - {e}")
    except Exception as e:
        print(f"[{idx+1}/{len(app_numbers)}] (코드: N/A) ⚠️ {app_num}: 처리 실패 - {e}")

    time.sleep(0.3)

# 3. 엑셀 저장
df = pd.DataFrame(results)
if not df.empty:
    filename = "KIPO_PUB_GAZETTE_LINKS.xlsx"
    df.to_excel(filename, index=False)
    print(f"\n총 {len(df)}건의 공개공보 링크를 추출했습니다.")
    files.download(filename)
else:
    print("\n저장된 공개공보 링크가 없습니다.")

공개공보 전용 조회를 시작합니다 (총 143건)

[1/143] (코드: 00) ✅ 1020257019800: 링크 확보!
[2/143] (코드: 00) ❌ 1020250155326: 데이터 없음 (태그는 정상이나 값이 비어있음)
[3/143] (코드: 00) ❌ 1020250115002: 데이터 없음 (태그는 정상이나 값이 비어있음)
[4/143] (코드: 00) ❌ 1020250077859: 데이터 없음 (태그는 정상이나 값이 비어있음)
[5/143] (코드: 00) ❌ 1020250077843: 데이터 없음 (태그는 정상이나 값이 비어있음)
[6/143] (코드: 00) ❌ 1020250068220: 데이터 없음 (태그는 정상이나 값이 비어있음)
[7/143] (코드: 00) ❌ 1020250051746: 데이터 없음 (태그는 정상이나 값이 비어있음)
[8/143] (코드: 00) ❌ 1020250045991: 데이터 없음 (태그는 정상이나 값이 비어있음)
[9/143] (코드: 00) ❌ 1020250034744: 데이터 없음 (태그는 정상이나 값이 비어있음)
[10/143] (코드: 00) ❌ 1020250027252: 데이터 없음 (태그는 정상이나 값이 비어있음)
[11/143] (코드: 00) ✅ 1020247023700: 링크 확보!
[12/143] (코드: 00) ✅ 1020247023307: 링크 확보!
[13/143] (코드: 00) ✅ 1020247011628: 링크 확보!
[14/143] (코드: 00) ❌ 1020240196542: 데이터 없음 (태그는 정상이나 값이 비어있음)
[15/143] (코드: 00) ❌ 1020240173044: 데이터 없음 (태그는 정상이나 값이 비어있음)
[16/143] (코드: 00) ❌ 1020240158222: 데이터 없음 (태그는 정상이나 값이 비어있음)
[17/143] (코드: 00) ❌ 1020240152277: 데이터 없음 (태그는 정상이나 값이 비어있음)
[18/143] (코드: 00) ❌ 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import requests
import io
import time
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload

# 1. 구글 드라이브 권한 인증
auth.authenticate_user()
drive_service = build('drive', 'v3')

# 2. 설정 정보
FOLDER_ID = '심사관폴더'  # 지정하신 드라이브 폴더 ID
EXCEL_FILE = 'KIPO_PUB_GAZETTE_LINKS.xlsx'       # 방금 생성된 엑셀 파일명

# 3. 엑셀 로드 및 업로드 루프
try:
    df = pd.read_excel(EXCEL_FILE)
    print(f"총 {len(df)}건의 파일 업로드를 시작합니다.\n")

    for idx, row in df.iterrows():
        app_num = str(row['출원번호'])
        pdf_url = row['공개공보링크']
        file_name = f"{app_num}_공개공보.pdf"

        try:
            # KIPRIS 서버에서 PDF 다운로드 (메모리로 로드)
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(pdf_url, headers=headers, timeout=60)

            if response.status_code == 200:
                # 구글 드라이브 업로드 설정
                file_metadata = {
                    'name': file_name,
                    'parents': [FOLDER_ID]
                }
                media = MediaIoBaseUpload(io.BytesIO(response.content), mimetype='application/pdf')

                # 업로드 실행
                uploaded_file = drive_service.files().create(
                    body=file_metadata,
                    media_body=media,
                    fields='id'
                ).execute()

                print(f"[{idx+1}/{len(df)}] ✅ {app_num} 업로드 성공 (ID: {uploaded_file.get('id')})")
            else:
                print(f"[{idx+1}/{len(df)}] ❌ {app_num} 다운로드 실패 (상태코드: {response.status_code})")

        except Exception as e:
            print(f"[{idx+1}/{len(df)}] ⚠️ {app_num} 처리 중 오류 발생: {e}")

        # 드라이브 API 할당량 준수 및 서버 부하 방지
        time.sleep(0.7)

    print("\n🎉 모든 공개공보 파일이 구글 드라이브로 이동되었습니다.")

except FileNotFoundError:
    print(f"🛑 에러: '{EXCEL_FILE}' 파일을 찾을 수 없습니다. 엑셀 추출이 먼저 완료되었는지 확인해주세요.")
except Exception as e:
    print(f"🛑 시스템 에러: {e}")

총 100건의 파일 업로드를 시작합니다.



[1/100] ✅ 1020257019800 업로드 성공 (ID: 1kIqsbpdaB6A-72eAHA7wGfZVoMHMdSXy)
[2/100] ✅ 1020247023700 업로드 성공 (ID: 1Nt9tLvY8vmikFPbIyGSk_Xxfo1JsYx0m)
[3/100] ✅ 1020247023307 업로드 성공 (ID: 1Lx7v-ij41-4WJsF_-Tw02qT267ps4ftE)
[4/100] ✅ 1020247011628 업로드 성공 (ID: 1pVaUZ3xUPur8sPNRDwnsZswQHKud_aMd)
[5/100] ✅ 1020240138555 업로드 성공 (ID: 1PAVLk3lLxwXN5qlGyo8Cuv2aGyidMdKA)
[6/100] ✅ 1020240058416 업로드 성공 (ID: 1szari3jjyjY_L5EerMaE1duD3e7yxHqe)
[7/100] ✅ 1020237015600 업로드 성공 (ID: 1b2YJiCm-xvI3UwYlr0HDIc9wCw9xNWK_)
[8/100] ✅ 1020237000315 업로드 성공 (ID: 11JOYrdebugHyjxjFxRSI0hSjp1xf_wvz)
[9/100] ✅ 1020230142119 업로드 성공 (ID: 1W32E-erpv3twrKi-FR_qrCrHPFlKWARG)
[10/100] ✅ 1020230134631 업로드 성공 (ID: 1xseBYEGIU3j8r6FMckXp-zQ9fFy09ttY)
[11/100] ✅ 1020230078104 업로드 성공 (ID: 1PCT85Q5AEPdSPfNrQcvBSFhxUDDfIs8R)
[12/100] ✅ 1020230061000 업로드 성공 (ID: 1UKT1n4XjQpBB97E2qD50-z4-3b64xmZ7)
[13/100] ✅ 1020230033546 업로드 성공 (ID: 1JusG17VN9xSHK2XLQEo-16726WbMsxGC)
[14/100] ✅ 1020230025066 업로드 성공 (ID: 1VpViwrebUmc0HXv6HeYSU9BCWHiqTDLn)
[

In [ ]:
import io
import time
import pandas as pd
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload, MediaIoBaseDownload

# 1. 라이브러리 설치 및 인증
!pip install -q pymupdf
import fitz  # PyMuPDF

auth.authenticate_user()
drive_service = build('drive', 'v3')

# 2. 대상 폴더 ID 설정
folder_ids = [
    '공개공보',  # 공개공보 폴더
    '심사관'   # 심사관 OA 폴더
]

def extract_text_from_pdf(pdf_bytes):
    """PDF 바이트 데이터를 받아 텍스트를 추출합니다."""
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

def process_folders(folders):
    for folder_id in folders:
        print(f"\n📂 폴더 처리 중: {folder_id}")

        # 폴더 내 PDF 파일 목록 가져오기
        query = f"'{folder_id}' in parents and mimeType='application/pdf' and trashed=false"
        results = drive_service.files().list(q=query, fields="files(id, name)").execute()
        files = results.get('files', [])

        print(f"총 {len(files)}개의 PDF를 발견했습니다.")

        for file in files:
            file_id = file['id']
            file_name = file['name']
            txt_file_name = file_name.rsplit('.', 1)[0] + ".txt"

            # 💡 동일한 이름의 txt 파일이 이미 있는지 확인 (중복 작업 방지)
            check_query = f"'{folder_id}' in parents and name='{txt_file_name}' and trashed=false"
            check_results = drive_service.files().list(q=check_query).execute()
            if check_results.get('files'):
                print(f"⏩ 스킵: {txt_file_name} (이미 존재함)")
                continue

            try:
                # 1. PDF 다운로드
                request = drive_service.files().get_media(fileId=file_id)
                fh = io.BytesIO()
                downloader = MediaIoBaseDownload(fh, request)
                done = False
                while not done:
                    _, done = downloader.next_chunk()

                # 2. 텍스트 추출
                print(f"📄 처리 중: {file_name}...", end="")
                pdf_text = extract_text_from_pdf(fh.getvalue())

                # 3. 텍스트 파일 업로드
                file_metadata = {
                    'name': txt_file_name,
                    'parents': [folder_id]
                }
                text_stream = io.BytesIO(pdf_text.encode('utf-8'))
                media = MediaIoBaseUpload(text_stream, mimetype='text/plain')

                drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()
                print(" ✅ 완료")

            except Exception as e:
                print(f" ❌ 에러 발생 ({file_name}): {e}")

            # API 할당량 준수
            time.sleep(0.5)

# 실행
process_folders(folder_ids)
print("\n🎉 모든 폴더의 텍스트 추출 작업이 완료되었습니다.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 63.1 MB/s eta 0:00:00



📂 폴더 처리 중: 1PSDwhVrWOuq_dr03egvq94KuPAh3LUr_
총 99개의 PDF를 발견했습니다.
📄 처리 중: 1020170054474_공개공보.pdf... ✅ 완료
📄 처리 중: 1020170083779_공개공보.pdf... ✅ 완료
📄 처리 중: 1020170144234_공개공보.pdf... ✅ 완료
📄 처리 중: 1020170155897_공개공보.pdf... ✅ 완료
📄 처리 중: 1020177006950_공개공보.pdf... ✅ 완료
📄 처리 중: 1020180025971_공개공보.pdf... ✅ 완료
📄 처리 중: 1020180031088_공개공보.pdf... ✅ 완료
📄 처리 중: 1020180063015_공개공보.pdf... ✅ 완료
📄 처리 중: 1020180116750_공개공보.pdf... ✅ 완료
📄 처리 중: 1020180123300_공개공보.pdf... ✅ 완료
📄 처리 중: 1020180125990_공개공보.pdf... ✅ 완료
📄 처리 중: 1020187037824_공개공보.pdf... ✅ 완료
📄 처리 중: 1020190101984_공개공보.pdf... ✅ 완료
📄 처리 중: 1020190164745_공개공보.pdf... ✅ 완료
📄 처리 중: 1020197012085_공개공보.pdf... ✅ 완료
📄 처리 중: 1020197013953_공개공보.pdf... ✅ 완료
📄 처리 중: 1020197015991_공개공보.pdf... ✅ 완료
📄 처리 중: 1020197026115_공개공보.pdf... ✅ 완료
📄 처리 중: 1020197037891_공개공보.pdf... ✅ 완료
📄 처리 중: 1020200001714_공개공보.pdf... ✅ 완료
📄 처리 중: 1020200001729_공개공보.pdf... ✅ 완료
📄 처리 중: 1020200004518_공개공보.pdf... ✅ 완료
📄 처리 중: 1020200059705_공개공보.pdf... ✅ 완료
📄 처리 중: 1020200093243_공개공보.pdf... ✅ 완

In [ ]:
import io
import re
import time
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload, MediaIoBaseDownload

# 1. 구글 드라이브 권한 인증
auth.authenticate_user()
drive_service = build('drive', 'v3')

# 2. 대상 폴더 ID 설정
FOLDER_ID = '공개공보'

def clean_claims_text(raw_text):
    """청구범위 추출 및 노이즈 제거 로직"""
    # 구간 추출 (청구범위 ~ 발명의 설명)
    start_marker = "청구범위"
    end_marker = "발명의 설명"

    start_idx = raw_text.find(start_marker)
    if start_idx == -1:
        start_idx = raw_text.find("청구항 1")

    end_idx = raw_text.find(end_marker)

    if start_idx != -1:
        content = raw_text[start_idx:end_idx] if end_idx != -1 else raw_text[start_idx:]
    else:
        return None # 구간을 찾지 못한 경우

    # 노이즈 제거
    # 1. 공개번호 (예: 공개특허 10-2019-0003202)
    content = re.sub(r'공개특허\s*\d{2}-\d{4}-\d{7}', '', content)
    # 2. 페이지 번호 (예: - 3 -, - 5 -)
    content = re.sub(r'-\s*\d+\s*-', '', content)
    # 3. 문서 헤더 단어 제거
    content = re.sub(r'^명세서', '', content)
    content = re.sub(r'^청구범위', '', content)
    # 4. 여러 줄 공백 정리
    content = re.sub(r'\n\s*\n', '\n', content).strip()

    return content

def process_and_save():
    print(f"📂 폴더 처리 시작: {FOLDER_ID}")

    # 폴더 내 모든 txt 파일 목록 가져오기
    query = f"'{FOLDER_ID}' in parents and mimeType='text/plain' and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])

    # 이미 정제된 파일명 목록 확보 (중복 방지)
    cleaned_filenames = [f['name'] for f in files if f['name'].endswith('_cleaned.txt')]

    # 정제 대상 파일만 필터링
    target_files = [f for f in files if f['name'].endswith('.txt') and not f['name'].endswith('_cleaned.txt')]

    print(f"총 {len(target_files)}개의 원본 텍스트 파일을 발견했습니다.")

    for file in target_files:
        file_id = file['id']
        file_name = file['name']
        new_file_name = file_name.replace(".txt", "_cleaned.txt")

        # 이미 정제된 파일이 있으면 스킵
        if new_file_name in cleaned_filenames:
            print(f"⏩ 스킵: {new_file_name} (이미 존재)")
            continue

        try:
            # 1. 원본 파일 다운로드
            request = drive_service.files().get_media(fileId=file_id)
            fh = io.BytesIO()
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()

            raw_text = fh.getvalue().decode('utf-8', errors='ignore')

            # 2. 텍스트 정제
            cleaned_text = clean_claims_text(raw_text)

            if cleaned_text:
                # 3. 정제된 텍스트 업로드
                file_metadata = {
                    'name': new_file_name,
                    'parents': [FOLDER_ID]
                }
                text_stream = io.BytesIO(cleaned_text.encode('utf-8'))
                media = MediaIoBaseUpload(text_stream, mimetype='text/plain')

                drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()
                print(f"✅ 완료: {new_file_name}")
            else:
                print(f"❌ 실패: {file_name} (청구범위 구간 미발견)")

        except Exception as e:
            print(f"⚠️ 에러: {file_name} 처리 중 오류 - {e}")

        time.sleep(0.3)

# 실행
process_and_save()
print("\n🎉 모든 파일의 전처리가 완료되었습니다.")

📂 폴더 처리 시작: 1PSDwhVrWOuq_dr03egvq94KuPAh3LUr_
총 99개의 원본 텍스트 파일을 발견했습니다.
✅ 완료: 1020257019800_공개공보_cleaned.txt
✅ 완료: 1020247023700_공개공보_cleaned.txt
✅ 완료: 1020247023307_공개공보_cleaned.txt
✅ 완료: 1020247011628_공개공보_cleaned.txt
✅ 완료: 1020240138555_공개공보_cleaned.txt
✅ 완료: 1020240058416_공개공보_cleaned.txt
✅ 완료: 1020237015600_공개공보_cleaned.txt
✅ 완료: 1020237000315_공개공보_cleaned.txt
✅ 완료: 1020230142119_공개공보_cleaned.txt
✅ 완료: 1020230134631_공개공보_cleaned.txt
✅ 완료: 1020230078104_공개공보_cleaned.txt
✅ 완료: 1020230061000_공개공보_cleaned.txt
✅ 완료: 1020230033546_공개공보_cleaned.txt
✅ 완료: 1020230025066_공개공보_cleaned.txt
✅ 완료: 1020230024571_공개공보_cleaned.txt
✅ 완료: 1020227027088_공개공보_cleaned.txt
✅ 완료: 1020227026698_공개공보_cleaned.txt
✅ 완료: 1020227017608_공개공보_cleaned.txt
✅ 완료: 1020227015892_공개공보_cleaned.txt
✅ 완료: 1020227015590_공개공보_cleaned.txt
✅ 완료: 1020227010826_공개공보_cleaned.txt
✅ 완료: 1020227009936_공개공보_cleaned.txt
✅ 완료: 1020227009700_공개공보_cleaned.txt
✅ 완료: 1020227009490_공개공보_cleaned.txt
✅ 완료: 1020227009135_공개공보_cleaned.txt
✅ 완

In [ ]:
import io
import time
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload, MediaIoBaseDownload

# 1. 라이브러리 설치 및 인증
!pip install -q pymupdf
import fitz  # PyMuPDF

auth.authenticate_user()
drive_service = build('drive', 'v3')

# 2. 대상 폴더 ID (심사관 OA 전용)
OA_FOLDER_ID = 'OA'

def extract_text_from_pdf(pdf_bytes):
    """PDF에서 텍스트 레이어를 고속으로 추출합니다."""
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

def process_oa_folder(folder_id):
    print(f"📂 OA 폴더 스캔 시작: {folder_id}")

    all_files = []
    page_token = None

    # 100개 제한 없이 모든 파일 리스트업
    while True:
        query = f"'{folder_id}' in parents and mimeType='application/pdf' and trashed=false"
        results = drive_service.files().list(
            q=query,
            fields="nextPageToken, files(id, name, size)",
            pageToken=page_token,
            pageSize=1000
        ).execute()

        all_files.extend(results.get('files', []))
        page_token = results.get('nextPageToken')
        if not page_token:
            break

    print(f"✅ 총 {len(all_files)}개의 PDF를 발견했습니다.")

    for idx, file in enumerate(all_files):
        file_id = file['id']
        file_name = file['name']
        file_size = int(file.get('size', 0))

        # 1KB 미만 파일(KIPRIS 다운로드 오류) 스킵
        if file_size < 1000:
            print(f"[{idx+1}/{len(all_files)}] ⚠️ {file_name}: 크기 이상({file_size}B). 스킵.")
            continue

        txt_file_name = file_name.rsplit('.', 1)[0] + ".txt"

        # 중복 작업 방지 (이미 txt가 있는지 확인)
        check = drive_service.files().list(
            q=f"'{folder_id}' in parents and name='{txt_file_name}' and trashed=false"
        ).execute()

        if check.get('files'):
            continue

        try:
            # 1. 다운로드
            request = drive_service.files().get_media(fileId=file_id)
            fh = io.BytesIO()
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()

            # 2. 텍스트 추출
            print(f"[{idx+1}/{len(all_files)}] 📄 {file_name} 처리 중...", end="")
            pdf_text = extract_text_from_pdf(fh.getvalue())

            # 3. 텍스트가 존재할 때만 저장
            if len(pdf_text.strip()) > 10:
                file_metadata = {'name': txt_file_name, 'parents': [folder_id]}
                text_stream = io.BytesIO(pdf_text.encode('utf-8'))
                media = MediaIoBaseUpload(text_stream, mimetype='text/plain')

                drive_service.files().create(body=file_metadata, media_body=media).execute()
                print(" ✅ 완료")
            else:
                print(" ❌ 실패 (텍스트 레이어 없음)")

        except Exception as e:
            print(f" ❌ 에러: {e}")

        time.sleep(0.2)

# 실행
process_oa_folder(OA_FOLDER_ID)
print("\n🎉 모든 OA 파일에 대한 처리가 완료되었습니다.")

📂 OA 폴더 스캔 시작: 1ZKJpdXnXmCjt_qvkIkaQClSyTqXOrZuo
✅ 총 138개의 PDF를 발견했습니다.
[9/138] ⚠️ 1020180025971.pdf: 크기 이상(21B). 스킵.
[56/138] ⚠️ 1020210037975.pdf: 크기 이상(21B). 스킵.
[87/138] ⚠️ 1020220091980.pdf: 크기 이상(21B). 스킵.
[95/138] ⚠️ 1020227008703.pdf: 크기 이상(19B). 스킵.
[99/138] ⚠️ 1020227010826.pdf: 크기 이상(19B). 스킵.
[101/138] 📄 1020227015892.pdf 처리 중... ✅ 완료
[102/138] 📄 1020227017608.pdf 처리 중... ✅ 완료
[103/138] 📄 1020227026698.pdf 처리 중... ✅ 완료
[104/138] 📄 1020227027088.pdf 처리 중... ✅ 완료
[105/138] 📄 1020230024571.pdf 처리 중... ✅ 완료
[106/138] 📄 1020230025066.pdf 처리 중... ✅ 완료
[107/138] 📄 1020230036605.pdf 처리 중... ✅ 완료
[108/138] 📄 1020230040630.pdf 처리 중... ✅ 완료
[109/138] 📄 1020230078104.pdf 처리 중... ✅ 완료
[111/138] 📄 1020230102592.pdf 처리 중... ✅ 완료
[112/138] 📄 1020230118745.pdf 처리 중... ✅ 완료
[113/138] 📄 1020230118769.pdf 처리 중... ✅ 완료
[114/138] 📄 1020230131860.pdf 처리 중... ✅ 완료
[115/138] 📄 1020230134631.pdf 처리 중... ✅ 완료
[116/138] 📄 1020230142119.pdf 처리 중... ✅ 완료
[118/138] 📄 1020237000315.pdf 처리 중... ✅ 완료
[119/1

In [ ]:
import io
import re
import time
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload, MediaIoBaseDownload

# 1. 구글 드라이브 인증
auth.authenticate_user()
drive_service = build('drive', 'v3')

FOLDER_ID = 'OA'

def extract_article_42(text):
    """
    어떤 변형 서식이든 42조만 악착같이 뽑아냅니다.
    """
    # 1. 시작점 유연화 (제목이 없으면 '아래' 또는 '이 출원은' 부터 시작)
    start_pattern = re.compile(r'\[?\s*구\s*체\s*적\s*인\s*거\s*절\s*이\s*유\s*\]?')
    start_match = start_pattern.search(text)

    if start_match:
        target_text = text[start_match.end():]
    else:
        # 💡 [해결] 제목이 없을 경우를 대비한 2차 방어선
        fallback = re.search(r'-\s*아\s*래\s*-|이\s*출\s*원\s*은', text)
        if fallback:
            target_text = text[fallback.start():]
        else:
            target_text = text # 이것도 없으면 전체를 대상으로 자르기 시작

    # 2. 💡 [해결] 끝단 절단기 (가장 먼저 걸리는 놈에서 무조건 자름)
    end_pattern = re.compile(
        r'(?:'
        r'끝\s*\.|'                                      # '끝.' (뒤에 공백이나 줄바꿈 상관없이 자름)
        r'20\d{2}\.\s*\d{1,2}\.\s*\d{1,2}\.|'            # '2025.07.14.' 심사관 날짜 서명란
        r'\[?\s*(?:참\s*고\s*사\s*항|심\s*사\s*관\s*의\s*조\s*치|거\s*절\s*이\s*유\s*를\s*극\s*복|첨\s*부)\s*\]?|'
        r'<+\s*안\s*내\s*>+'                             # '<< 안내 >>'
        r')'
    )

    end_match = end_pattern.search(target_text)
    if end_match:
        target_text = target_text[:end_match.start()]

    # 3. 잔여 노이즈 청소
    target_text = re.sub(r'수신\s*:.*?\n', '', target_text)
    target_text = re.sub(r'\d{2}-\d{4}-\d{7}\n?', '', target_text)
    target_text = re.sub(r'\d+/\d+\n?', '', target_text)
    target_text = re.sub(r'-\s*\d+\s*-\n?', '', target_text)

    target_text = target_text.strip()

    # 4. 💡 [해결] 블록 분할 ('1.' 뿐만 아니라 '1)' 형식도 완벽 호환)
    blocks = re.split(r'(?=\n\s*\d+\s*[\.\)]\s)', '\n' + target_text)

    art42_extracted = []
    keep_flag = False

    for block in blocks:
        if not block.strip(): continue
        has_42 = bool(re.search(r'42\s*조|제\s*42\s*조', block))
        has_other_law = bool(re.search(r'29\s*조|33\s*조|제\s*29\s*조|제\s*33\s*조', block))

        if has_42: keep_flag = True
        elif has_other_law and not has_42: keep_flag = False

        if keep_flag: art42_extracted.append(block.strip())

    if not art42_extracted and bool(re.search(r'42\s*조|제\s*42\s*조', target_text)):
        return target_text.strip()

    return "\n\n".join(art42_extracted) if art42_extracted else None

def process_oa_hard_update():
    print(f"📂 [하드 덮어쓰기 모드] 42조 핀셋 정제 시작 (폴더: {FOLDER_ID})")

    all_files = []
    page_token = None

    while True:
        query = f"'{FOLDER_ID}' in parents and mimeType='text/plain' and trashed=false"
        results = drive_service.files().list(
            q=query, fields="nextPageToken, files(id, name)", pageToken=page_token, pageSize=1000
        ).execute()
        all_files.extend(results.get('files', []))
        page_token = results.get('nextPageToken')
        if not page_token: break

    raw_files = [f for f in all_files if f['name'].endswith('.txt') and not f['name'].endswith('_cleaned.txt')]
    cleaned_dict = {f['name']: f['id'] for f in all_files if f['name'].endswith('_cleaned.txt')}

    print(f"✅ 총 {len(raw_files)}건 검사 및 재작성 중...")

    for idx, file in enumerate(raw_files):
        file_id = file['id']
        file_name = file['name']
        new_file_name = file_name.replace(".txt", "_cleaned.txt")

        try:
            request = drive_service.files().get_media(fileId=file_id)
            fh = io.BytesIO()
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done: _, done = downloader.next_chunk()

            raw_content = fh.getvalue().decode('utf-8', errors='ignore')
            final_text = extract_article_42(raw_content)

            if final_text:
                media = MediaIoBaseUpload(io.BytesIO(final_text.encode('utf-8')), mimetype='text/plain')

                # 💡 [해결] 기존 파일을 아예 '삭제'하고 새로 만들어버림 (캐시 꼬임 방지)
                if new_file_name in cleaned_dict:
                    existing_id = cleaned_dict[new_file_name]
                    drive_service.files().delete(fileId=existing_id).execute()

                file_metadata = {'name': new_file_name, 'parents': [FOLDER_ID]}
                drive_service.files().create(body=file_metadata, media_body=media).execute()
                print(f"[{idx+1}/{len(raw_files)}] 🔄 완벽 교체 완료: {new_file_name}")
            else:
                print(f"[{idx+1}/{len(raw_files)}] ⏩ 42조 없음 스킵: {file_name}")

        except Exception as e:
            print(f"[{idx+1}/{len(raw_files)}] ❌ {file_name} 에러: {e}")

        time.sleep(0.3)

# 실행
process_oa_hard_update()
print("\n🎉 모든 파일 강제 업데이트 완료! 이제 진짜 끝. 입니다!")

📂 [하드 덮어쓰기 모드] 42조 핀셋 정제 시작 (폴더: 1ZKJpdXnXmCjt_qvkIkaQClSyTqXOrZuo)
✅ 총 118건 검사 및 재작성 중...
[1/118] 🔄 완벽 교체 완료: 1020257019800_cleaned.txt
[2/118] 🔄 완벽 교체 완료: 1020250115002_cleaned.txt
[3/118] 🔄 완벽 교체 완료: 1020250077859_cleaned.txt
[4/118] 🔄 완벽 교체 완료: 1020250077843_cleaned.txt
[5/118] ⏩ 42조 없음 스킵: 1020250068220.txt
[6/118] ⏩ 42조 없음 스킵: 1020250034744.txt
[7/118] ⏩ 42조 없음 스킵: 1020250027252.txt
[8/118] 🔄 완벽 교체 완료: 1020247023700_cleaned.txt
[9/118] 🔄 완벽 교체 완료: 1020247023307_cleaned.txt
[10/118] ⏩ 42조 없음 스킵: 1020247011628.txt
[11/118] 🔄 완벽 교체 완료: 1020240196542_cleaned.txt
[12/118] ⏩ 42조 없음 스킵: 1020240173044.txt
[13/118] 🔄 완벽 교체 완료: 1020240152277_cleaned.txt
[14/118] 🔄 완벽 교체 완료: 1020240138555_cleaned.txt
[15/118] 🔄 완벽 교체 완료: 1020240081489_cleaned.txt
[16/118] ⏩ 42조 없음 스킵: 1020240058416.txt
[17/118] 🔄 완벽 교체 완료: 1020240026694_cleaned.txt
[18/118] 🔄 완벽 교체 완료: 1020237015600_cleaned.txt
[19/118] 🔄 완벽 교체 완료: 1020237000315_cleaned.txt
[20/118] 🔄 완벽 교체 완료: 1020230142119_cleaned.txt
[21/118] 🔄 완벽 교체 완료:

In [ ]:
import io
import re
import time
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload, MediaIoBaseDownload

# 1. 구글 드라이브 인증
auth.authenticate_user()
drive_service = build('drive', 'v3')

# 2. 대상 폴더 ID (등록공보 폴더)
FOLDER_ID = '등록공보'

def extract_clean_claims(raw_text):
    """
    등록공보의 청구범위만 추출하고 모든 노이즈를 제거합니다.
    """
    # 1. 전역 노이즈 사전 제거 (페이지 구분선, 등록번호, 페이지 번호 등)
    # --- [페이지 구분] --- 형태 제거
    text = re.sub(r'[-]+\s*\[\s*페\s*이\s*지\s*구\s*분\s*\]\s*[-]+', '', raw_text)
    # 등록특허 10-xxxxxxx 제거
    text = re.sub(r'등\s*록\s*특\s*허\s*\d{2}-\d{7}', '', text)
    # - 2 -, - 3 - 같은 하단 페이지 번호 제거
    text = re.sub(r'-\s*\d+\s*-', '', text)
    # 문서 상단의 '명 세 서' 단어 제거
    text = re.sub(r'명\s*세\s*서', '', text)

    # 2. 구간 슬라이싱 (청구범위 ~ 발명의 설명)
    start_pattern = re.compile(r'청\s*구\s*범\s*위|청\s*구\s*항\s*1')
    end_pattern = re.compile(r'발\s*명\s*의\s*설\s*명')

    start_match = start_pattern.search(text)
    if not start_match:
        return None # 청구항 시작점을 찾지 못함

    target_text = text[start_match.start():]

    end_match = end_pattern.search(target_text)
    if end_match:
        target_text = target_text[:end_match.start()]

    # 3. 최상단 '청구범위' 텍스트 자체를 지우고 싶다면 아래 주석 해제
    target_text = re.sub(r'^청\s*구\s*범\s*위\s*\n', '', target_text.strip())

    # 4. 불필요한 다중 공백 및 여러 줄 개행 정리 (데이터셋 품질 향상)
    target_text = re.sub(r'\n\s*\n', '\n', target_text)

    return target_text.strip()

def process_registered_patents():
    print(f"📂 [등록공보 청구항 추출] 작업을 시작합니다. (폴더: {FOLDER_ID})")

    all_files = []
    page_token = None

    # 100건 제한을 풀고 폴더 내 전체 텍스트 파일 리스트업
    while True:
        query = f"'{FOLDER_ID}' in parents and mimeType='text/plain' and trashed=false"
        results = drive_service.files().list(
            q=query, fields="nextPageToken, files(id, name)", pageToken=page_token, pageSize=1000
        ).execute()
        all_files.extend(results.get('files', []))
        page_token = results.get('nextPageToken')
        if not page_token: break

    # 원본 파일(.txt)과 이미 처리된 파일(_extracted.txt) 분리
    extracted_dict = {f['name']: f['id'] for f in all_files if f['name'].endswith('_extracted.txt')}
    raw_files = [f for f in all_files if f['name'].endswith('.txt') and not f['name'].endswith('_extracted.txt') and not f['name'].endswith('_cleaned.txt')]

    print(f"✅ 총 {len(raw_files)}개의 원본 파일을 정제합니다.\n")

    for idx, file in enumerate(raw_files):
        file_id = file['id']
        file_name = file['name']
        new_file_name = file_name.replace(".txt", "_extracted.txt")

        try:
            # 1. 텍스트 다운로드
            request = drive_service.files().get_media(fileId=file_id)
            fh = io.BytesIO()
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done: _, done = downloader.next_chunk()

            raw_content = fh.getvalue().decode('utf-8', errors='ignore')

            # 2. 청구항 추출 로직 적용
            extracted_text = extract_clean_claims(raw_content)

            if extracted_text:
                media = MediaIoBaseUpload(io.BytesIO(extracted_text.encode('utf-8')), mimetype='text/plain')

                # 3. 덮어쓰기 또는 신규 생성
                if new_file_name in extracted_dict:
                    existing_id = extracted_dict[new_file_name]
                    drive_service.files().update(fileId=existing_id, media_body=media).execute()
                    print(f"[{idx+1}/{len(raw_files)}] 🔄 덮어쓰기 완료: {new_file_name}")
                else:
                    file_metadata = {'name': new_file_name, 'parents': [FOLDER_ID]}
                    drive_service.files().create(body=file_metadata, media_body=media).execute()
                    print(f"[{idx+1}/{len(raw_files)}] ✅ 신규 생성 완료: {new_file_name}")
            else:
                print(f"[{idx+1}/{len(raw_files)}] ⏩ 청구항 없음 스킵: {file_name}")

        except Exception as e:
            print(f"[{idx+1}/{len(raw_files)}] ❌ {file_name} 처리 중 에러: {e}")

        time.sleep(0.2)

# 코드 실행
process_registered_patents()
print("\n🎉 모든 등록공보 파일의 청구항 추출이 성공적으로 완료되었습니다!")

📂 [등록공보 청구항 추출] 작업을 시작합니다. (폴더: 1kXj88945CvG_jkvZlI2DIjAN3CL7qFJj)
✅ 총 143개의 원본 파일을 정제합니다.

[1/143] ✅ 신규 생성 완료: 1020220071866_extracted.txt
[2/143] ✅ 신규 생성 완료: 1020217023623_extracted.txt
[3/143] ✅ 신규 생성 완료: 1020210161676_extracted.txt
[4/143] ✅ 신규 생성 완료: 1020217033033_extracted.txt
[5/143] ✅ 신규 생성 완료: 1020220091980_extracted.txt
[6/143] ✅ 신규 생성 완료: 1020207018024_extracted.txt
[7/143] ✅ 신규 생성 완료: 1020227015892_extracted.txt
[8/143] ✅ 신규 생성 완료: 1020220033925_extracted.txt
[9/143] ✅ 신규 생성 완료: 1020180031088_extracted.txt
[10/143] ✅ 신규 생성 완료: 1020250077859_extracted.txt
[11/143] ✅ 신규 생성 완료: 1020217018412_extracted.txt
[12/143] ✅ 신규 생성 완료: 1020220104033_extracted.txt
[13/143] ✅ 신규 생성 완료: 1020190164745_extracted.txt
[14/143] ✅ 신규 생성 완료: 1020250115002_extracted.txt
[15/143] ✅ 신규 생성 완료: 1020227027088_extracted.txt
[16/143] ✅ 신규 생성 완료: 1020200004518_extracted.txt
[17/143] ✅ 신규 생성 완료: 1020210057704_extracted.txt
[18/143] ✅ 신규 생성 완료: 1020240158222_extracted.txt
[19/143] ✅ 신규 생성 완료: 1020180116750_

In [ ]:
import io
import json
import re
import time
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# 1. 구글 드라이브 인증
auth.authenticate_user()
drive_service = build('drive', 'v3')

# 2. 폴더 ID 설정
FOLDER_1_PUB = ''   # 공개공보 청구항
FOLDER_2_OA  = ''   # OA 42조
FOLDER_3_REG = ''   # 등록공보 청구항

# 시스템 프롬프트 (간결한 버전 유지)
SYSTEM_PROMPT = (
    "당신은 대한민국 특허청의 엄격하고 논리적인 심사관입니다. "
    "제시된 특허 청구범위를 검토하여 특허법 제42조(기재불비) 위반 여부를 심사하세요. "
    "위반 사항이 있다면 조항과 이유를 구체적으로 지적하고, 문제가 없다면 등록 가능하다고 답변하세요."
)

def get_target_file_list(folder_id, target_suffix):
    """
    폴더 내 파일 중 '특정 확장자(suffix)'로 끝나는 파일만 정확하게 골라냅니다.
    원본 txt와 섞이는 문제를 원천 차단합니다.
    """
    files_dict = {}
    page_token = None
    while True:
        query = f"'{folder_id}' in parents and mimeType='text/plain' and trashed=false"
        results = drive_service.files().list(
            q=query, fields="nextPageToken, files(id, name)", pageToken=page_token, pageSize=1000
        ).execute()

        for f in results.get('files', []):
            # 💡 [핵심 수정] 지정한 꼬리표(_cleaned.txt 등)가 붙은 파일만 취급
            if f['name'].endswith(target_suffix):
                # 출원번호 13자리 추출
                match = re.search(r'^(\d{13})', f['name'])
                if match:
                    app_num = match.group(1)
                    files_dict[app_num] = {'id': f['id'], 'name': f['name']}

        page_token = results.get('nextPageToken')
        if not page_token: break
    return files_dict

def download_text(file_id):
    request = drive_service.files().get_media(fileId=file_id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    return fh.getvalue().decode('utf-8', errors='ignore').strip()

def build_advanced_jsonl():
    print("📂 각 폴더의 [정제 완료된 파일]만 선별하여 불러오는 중...")

    # 각 폴더별로 정확한 목표 파일의 꼬리표(suffix)를 지정합니다.
    pub_files = get_target_file_list(FOLDER_1_PUB, '_cleaned.txt') # 1번: 공개공보는 _cleaned.txt
    oa_files  = get_target_file_list(FOLDER_2_OA, '_cleaned.txt')  # 2번: OA도 _cleaned.txt
    reg_files = get_target_file_list(FOLDER_3_REG, '_extracted.txt') # 3번: 등록공보는 _extracted.txt

    dataset = []

    # ---------------------------------------------------------
    # [제1예시] 거절 케이스: 1번(공개공보) -> 2번(OA)
    # ---------------------------------------------------------
    print(f"\n[제1예시] 거절 케이스 데이터 조립 시작 (목표: {len(oa_files)}건)")
    matched_oa_count = 0
    for app_num, oa_info in oa_files.items():
        if app_num in pub_files:
            try:
                user_text = download_text(pub_files[app_num]['id'])
                agent_text = download_text(oa_info['id'])

                if user_text and agent_text:
                    dataset.append({
                        "messages": [
                            {"role": "system", "content": SYSTEM_PROMPT},
                            {"role": "user", "content": f"다음 청구범위를 심사하라:\n\n{user_text}"},
                            {"role": "assistant", "content": agent_text}
                        ]
                    })
                    matched_oa_count += 1
            except Exception as e:
                print(f"⚠️ {app_num} 조립 에러: {e}")
            time.sleep(0.2)

    # ---------------------------------------------------------
    # [제2예시] 등록 케이스: 3번(등록공보) -> "등록 가능"
    # ---------------------------------------------------------
    print(f"\n[제2예시] 등록 케이스 데이터 조립 시작 (목표: {len(reg_files)}건)")
    matched_reg_count = 0
    for app_num, reg_info in reg_files.items():
        try:
            user_text = download_text(reg_info['id'])

            if user_text:
                dataset.append({
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": f"다음 청구범위를 심사하라:\n\n{user_text}"},
                        {"role": "assistant", "content": "검토 결과, 특허법 제42조에 따른 기재불비 사항이 발견되지 않았습니다. 등록 가능합니다."}
                    ]
                })
                matched_reg_count += 1
        except Exception as e:
            print(f"⚠️ {app_num} 조립 에러: {e}")
        time.sleep(0.2)

    # ---------------------------------------------------------
    # JSONL 파일 저장
    # ---------------------------------------------------------
    output_filename = "finetuning_dataset.jsonl"
    with open(output_filename, 'w', encoding='utf-8') as f:
        for data in dataset:
            f.write(json.dumps(data, ensure_ascii=False) + '\n')

    print(f"\n🎉 작업 완료!")
    print(f"- [제1예시] 거절(OA) 매칭: {matched_oa_count}건")
    print(f"- [제2예시] 등록(정상) 매칭: {matched_reg_count}건")
    print(f"- 총 {len(dataset)}건의 데이터가 '{output_filename}'에 저장되었습니다.")

    from google.colab import files
    files.download(output_filename)

# 실행
build_advanced_jsonl()

📂 각 폴더의 [정제 완료된 파일]만 선별하여 불러오는 중...

[제1예시] 거절 케이스 데이터 조립 시작 (목표: 75건)

[제2예시] 등록 케이스 데이터 조립 시작 (목표: 143건)

🎉 작업 완료!
- [제1예시] 거절(OA) 매칭: 55건
- [제2예시] 등록(정상) 매칭: 143건
- 총 198건의 데이터가 'finetuning_dataset.jsonl'에 저장되었습니다.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
import random

def create_advanced_split(input_file="finetuning_dataset.jsonl", train_file="train.jsonl", val_file="val.jsonl", oversample_rate=2):
    print("🚀 데이터 분할 및 오버샘플링 작업을 시작합니다...")

    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            lines = f.readlines()
    except FileNotFoundError:
        print("❌ 에러: 'finetuning_dataset.jsonl' 파일을 찾을 수 없습니다. 먼저 조립 코드를 실행해 주세요.")
        return

    rej_data = [] # 거절(OA) 데이터
    reg_data = [] # 등록(정상) 데이터

    # 1. 데이터 분류 (Assistant의 답변 내용으로 구분)
    for line in lines:
        data = json.loads(line)
        assistant_reply = data['messages'][2]['content']

        # '등록 가능합니다'라는 문구가 있으면 등록 데이터, 아니면 거절 데이터
        if "등록 가능합니다" in assistant_reply:
            reg_data.append(line)
        else:
            rej_data.append(line)

    print(f"📊 분류 완료: 거절 {len(rej_data)}건 / 등록 {len(reg_data)}건")

    # 2. 랜덤 셔플 (골고루 섞기)
    random.seed(42) # 재현성을 위해 시드 고정
    random.shuffle(rej_data)
    random.shuffle(reg_data)

    # 3. 9:1 분할 (Split) - 💡 오버샘플링 전에 분할해야 평가가 정확해집니다!
    rej_split_idx = int(len(rej_data) * 0.9)
    reg_split_idx = int(len(reg_data) * 0.9)

    rej_train = rej_data[:rej_split_idx]
    rej_val = rej_data[rej_split_idx:]

    reg_train = reg_data[:reg_split_idx]
    reg_val = reg_data[reg_split_idx:]

    # 4. Train 데이터 오버샘플링 (거절 데이터를 설정한 배수만큼 뻥튀기)
    rej_train_oversampled = rej_train * oversample_rate

    # 5. 최종 데이터 병합 및 셔플
    train_data = rej_train_oversampled + reg_train
    val_data = rej_val + reg_val

    random.shuffle(train_data)
    random.shuffle(val_data)

    # 6. 파일 저장
    with open(train_file, 'w', encoding='utf-8') as f:
        f.writelines(train_data)

    with open(val_file, 'w', encoding='utf-8') as f:
        f.writelines(val_data)

    print(f"\n✅ 완벽하게 분할되었습니다!")
    print(f"🎯 Train (학습용): {len(train_data)}건 (거절 {len(rej_train_oversampled)}건 + 등록 {len(reg_train)}건)")
    print(f"🧪 Validation (검증용): {len(val_data)}건 (거절 {len(rej_val)}건 + 등록 {len(reg_val)}건)")

    # 7. 코랩에서 자동 다운로드
    try:
        from google.colab import files
        files.download(train_file)
        files.download(val_file)
    except Exception as e:
        print("다운로드 창을 띄우는 중 문제가 발생했습니다. 좌측 파일 탐색기에서 직접 다운로드해 주세요.")

# 실행
create_advanced_split()

🚀 데이터 분할 및 오버샘플링 작업을 시작합니다...
📊 분류 완료: 거절 55건 / 등록 143건

✅ 완벽하게 분할되었습니다!
🎯 Train (학습용): 226건 (거절 98건 + 등록 128건)
🧪 Validation (검증용): 21건 (거절 6건 + 등록 15건)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>